In [15]:
import numpy as np
from dash import Dash, dcc, html, Input, Output
import plotly.graph_objects as go


# ============================================================
# Create the Dash web application
# ============================================================

# Dash is the framework that makes the interactive web app.
# app is the main object that stores the layout and callbacks.
app = Dash(__name__)


# ============================================================
# Helper functions
# ============================================================

def current_sign(val):
    """
    This function converts the current direction into a sign.

    If the user chooses Positive, the value is +1.
    If the user chooses Negative, the value is -1.

    We use this sign to reverse the current direction.
    Reversing the current should also reverse the magnetic field direction.
    """

    if isinstance(val, str):
        return -1 if val.lower().startswith("neg") else 1

    return -1 if val < 0 else 1


def add_arrow_on_curve(fig, pts, color="rgba(0,70,220,1)", size=0.15, width=5):
    """
    This function places a small arrow on a field line.

    fig:
        The Plotly figure we are drawing on.

    pts:
        A list of points along one magnetic field line.

    The arrow is placed near the middle of the curve.
    The direction of the arrow follows the direction of the curve.
    """

    # If the curve has too few points, we cannot safely draw an arrow.
    if len(pts) < 5:
        return

    # Choose the middle point of the field line.
    i = len(pts) // 2

    # p is the starting point of the arrow.
    p = pts[i]

    # d is the direction of the arrow.
    # We estimate the tangent direction using nearby points.
    d = pts[min(i + 1, len(pts) - 1)] - pts[max(i - 1, 0)]

    # Draw the actual arrow.
    add_simple_arrow(fig, p, d, color=color, size=size, wing=0.05, width=width)


# ============================================================
# Wire geometry
# ============================================================

def make_wire(shape, radius=1.0, length=4.0, n=400, turns=6):
    """
    This function creates the coordinates of the wire.

    The wire is represented by many small points:
        x[0], y[0], z[0]
        x[1], y[1], z[1]
        x[2], y[2], z[2]
        ...

    The Biot-Savart law is then applied to tiny line segments
    between neighboring points.

    n:
        Number of points used to build the wire.
        Larger n gives a smoother wire, but it also makes the code slower.
    """

    # --------------------------------------------------------
    # Straight wire
    # --------------------------------------------------------
    if shape == "straight":

        # A straight wire is placed along the z-axis.
        # x = 0 and y = 0 for every point.
        # z changes from -length/2 to +length/2.
        z = np.linspace(-length / 2, length / 2, n)
        x = np.zeros_like(z)
        y = np.zeros_like(z)

    # --------------------------------------------------------
    # Circular loop
    # --------------------------------------------------------
    elif shape == "circle":

        # A circle is described by an angle t from 0 to 2π.
        t = np.linspace(0, 2 * np.pi, n)

        # Standard circle equations in the xy-plane:
        # x = R cos(t)
        # y = R sin(t)
        # z = 0
        x = radius * np.cos(t)
        y = radius * np.sin(t)
        z = np.zeros_like(t)

    # --------------------------------------------------------
    # Arc
    # --------------------------------------------------------
    elif shape == "arc":

        # This is only a half circle, from -π/2 to +π/2.
        t = np.linspace(-np.pi / 2, np.pi / 2, n)

        x = radius * np.cos(t)
        y = radius * np.sin(t)
        z = np.zeros_like(t)

    # --------------------------------------------------------
    # Square loop
    # --------------------------------------------------------
    elif shape == "square":

        # The square is centered at the origin.
        # The value a is the half-side length.
        a = radius

        # Divide the total number of points into four sides.
        m = n // 4

        # Bottom side: y = -a, x goes from -a to +a
        x1 = np.linspace(-a, a, m)
        y1 = -a * np.ones_like(x1)

        # Right side: x = +a, y goes from -a to +a
        y2 = np.linspace(-a, a, m)
        x2 = a * np.ones_like(y2)

        # Top side: y = +a, x goes from +a to -a
        x3 = np.linspace(a, -a, m)
        y3 = a * np.ones_like(x3)

        # Left side: x = -a, y goes from +a to -a
        y4 = np.linspace(a, -a, n - 3 * m)
        x4 = -a * np.ones_like(y4)

        # Put the four sides together into one continuous wire.
        x = np.concatenate([x1, x2, x3, x4])
        y = np.concatenate([y1, y2, y3, y4])
        z = np.zeros_like(x)

    # --------------------------------------------------------
    # Solenoid
    # --------------------------------------------------------
    elif shape == "solenoid":

        # A solenoid is a helix.
        # It winds around while moving along the z-axis.
        t = np.linspace(0, 2 * np.pi * turns, n)

        x = radius * np.cos(t)
        y = radius * np.sin(t)
        z = np.linspace(-length / 2, length / 2, n)

    else:
        raise ValueError("Unknown shape")

    return x, y, z


# ============================================================
# Biot-Savart magnetic field calculation
# ============================================================

def field_at_point(px, py, pz, wire_x, wire_y, wire_z, current):
    """
    This function calculates the magnetic field B at one point P.

    P has coordinates:
        px, py, pz

    The wire is broken into many small pieces.

    For each small piece of wire, we apply the Biot-Savart idea:

        dB is proportional to I dL x r / r^3

    We ignore the constant μ0 / 4π because this is a visualization.
    The important part here is the direction and shape of the field.
    """

    # Start with zero magnetic field.
    bx = by = bz = 0.0

    # Loop over all small wire segments.
    for i in range(len(wire_x) - 1):

        # First endpoint of the small wire segment.
        x1, y1, z1 = wire_x[i], wire_y[i], wire_z[i]

        # Second endpoint of the small wire segment.
        x2, y2, z2 = wire_x[i + 1], wire_y[i + 1], wire_z[i + 1]

        # dL vector points along the current element.
        dlx = x2 - x1
        dly = y2 - y1
        dlz = z2 - z1

        # Use the midpoint of the small segment as the source point.
        xm = 0.5 * (x1 + x2)
        ym = 0.5 * (y1 + y2)
        zm = 0.5 * (z1 + z2)

        # r vector points from the current element to the observation point P.
        rx = px - xm
        ry = py - ym
        rz = pz - zm

        # Compute r squared.
        # The tiny number avoids division by zero if P is too close to the wire.
        r2 = rx**2 + ry**2 + rz**2 + 1e-12

        # Compute r.
        r = np.sqrt(r2)

        # We need r^3 in the Biot-Savart law.
        r3 = r2 * r

        # Cross product dL x r.
        # This determines the direction of the magnetic field contribution.
        cx = dly * rz - dlz * ry
        cy = dlz * rx - dlx * rz
        cz = dlx * ry - dly * rx

        # Add this small contribution to the total field.
        bx += current * cx / r3
        by += current * cy / r3
        bz += current * cz / r3

    return np.array([bx, by, bz], dtype=float)


# ============================================================
# Field-line tracing functions
# ============================================================

def trace_generic_field_line(seed, wire_x, wire_y, wire_z, current, step=0.045, nsteps=320, box=3.0):
    """
    This function traces one magnetic field line.

    seed:
        Starting point of the field line.

    The idea:
        At each point, calculate B.
        Move a small step in the direction of B.
        Repeat many times.

    This creates a curve that follows the magnetic field.
    """

    pts = [np.array(seed, dtype=float)]

    for _ in range(nsteps):

        # Current point on the field line.
        p = pts[-1]

        # Magnetic field at that point.
        B = field_at_point(p[0], p[1], p[2], wire_x, wire_y, wire_z, current)

        # Magnitude of B.
        bmag = np.linalg.norm(B)

        # If the field is almost zero, stop tracing.
        if bmag < 1e-10:
            break

        # Move one small step in the direction of B.
        p_new = p + step * B / bmag

        # Stop if the point leaves the viewing box.
        if np.max(np.abs(p_new)) > box:
            break

        pts.append(p_new)

    return np.array(pts)


def trace_meridian_line(seed_x, seed_z, wire_x, wire_y, wire_z, current, step=0.045, nsteps=340, box=3.0):
    """
    This traces a field line in the xz-plane.

    This is useful for circular loops and solenoids,
    because their fields have rotational symmetry.

    We trace a curve in one plane, then rotate it around an axis.
    """

    pts = [np.array([seed_x, 0.0, seed_z], dtype=float)]

    for _ in range(nsteps):

        p = pts[-1]

        B = field_at_point(p[0], p[1], p[2], wire_x, wire_y, wire_z, current)

        # Keep only the x and z components.
        # This forces the line to stay in the meridian plane.
        v = np.array([B[0], 0.0, B[2]], dtype=float)

        vmag = np.linalg.norm(v)

        if vmag < 1e-10:
            break

        p_new = p + step * v / vmag

        if np.max(np.abs(p_new)) > box:
            break

        pts.append(p_new)

    return np.array(pts)


def rotate_xz_curve(curve_xz, phi):
    """
    Rotate a curve around the z-axis.

    This is used for the circular loop.
    The loop lies in the xy-plane.
    Its symmetry axis is the z-axis.
    """

    c = np.cos(phi)
    s = np.sin(phi)

    x = curve_xz[:, 0]
    z = curve_xz[:, 2]

    xr = x * c
    yr = x * s
    zr = z

    return np.column_stack([xr, yr, zr])


def rotate_xz_curve_about_x(curve_xz, phi):
    """
    Rotate a curve around the x-axis.

    This is used for the solenoid visualization.
    """

    c = np.cos(phi)
    s = np.sin(phi)

    x = curve_xz[:, 0]
    z = curve_xz[:, 2]

    xr = x
    yr = z * s
    zr = z * c

    return np.column_stack([xr, yr, zr])


# ============================================================
# Arrow drawing
# ============================================================

def add_simple_arrow(fig, p, d, color="rgba(0,90,255,1)", size=0.16, wing=0.05, width=5):
    """
    Draw a simple 3D arrow.

    p:
        Starting point of the arrow.

    d:
        Direction of the arrow.

    The arrow is drawn using line segments:
        one line for the shaft
        two short lines for the arrowhead
    """

    d = np.array(d, dtype=float)
    n = np.linalg.norm(d)

    if n < 1e-12:
        return

    # Normalize the direction vector.
    d = d / n

    p = np.array(p, dtype=float)

    # Tip of the arrow.
    tip = p + size * d

    # Choose a reference vector to build the arrowhead.
    ref = np.array([0.0, 0.0, 1.0])

    # If d is too close to the z direction, choose another reference.
    if abs(np.dot(d, ref)) > 0.9:
        ref = np.array([1.0, 0.0, 0.0])

    # side points sideways relative to the arrow direction.
    side = np.cross(d, ref)
    side_n = np.linalg.norm(side)

    if side_n < 1e-12:
        return

    side = side / side_n

    # Two sides of the arrowhead.
    left = tip - 0.45 * size * d + wing * side
    right = tip - 0.45 * size * d - wing * side

    # Draw arrow shaft.
    fig.add_trace(go.Scatter3d(
        x=[p[0], tip[0]],
        y=[p[1], tip[1]],
        z=[p[2], tip[2]],
        mode="lines",
        line=dict(color=color, width=width),
        showlegend=False,
        hoverinfo="skip"
    ))

    # Draw arrowhead.
    fig.add_trace(go.Scatter3d(
        x=[left[0], tip[0], right[0]],
        y=[left[1], tip[1], right[1]],
        z=[left[2], tip[2], right[2]],
        mode="lines",
        line=dict(color=color, width=width),
        showlegend=False,
        hoverinfo="skip"
    ))


def add_wire_current_arrow(fig, shape, wire_x, wire_y, wire_z, current_dir):
    """
    Draw a black arrow on the red wire.

    This arrow shows the direction of the electric current.
    """

    sgn = current_sign(current_dir)
    n = len(wire_x)

    if n < 3:
        return

    # Choose where to place the arrow depending on the shape.
    if shape == "straight":
        i = n // 2
    elif shape == "circle":
        i = n // 4
    elif shape == "square":
        i = n // 8
    elif shape == "arc":
        i = n // 2
    elif shape == "solenoid":
        i = n // 3
    else:
        i = n // 2

    # Make sure i is safely inside the array.
    i = max(1, min(n - 2, i))

    # Point where the arrow starts.
    p = np.array([wire_x[i], wire_y[i], wire_z[i]])

    # Direction of the wire at that point.
    d = np.array([
        wire_x[i + 1] - wire_x[i - 1],
        wire_y[i + 1] - wire_y[i - 1],
        wire_z[i + 1] - wire_z[i - 1]
    ]) * sgn

    add_simple_arrow(fig, p, d, color="black", size=0.35, wing=0.1, width=8)


# ============================================================
# Magnetic field for a straight wire
# ============================================================

def add_straight_wire_field(fig, current_mag, current_dir, length):
    """
    Draw circular magnetic field lines around a straight wire.

    A straight current-carrying wire produces circular magnetic field lines.
    The direction follows the right-hand rule.
    """

    line_color = "rgba(0,140,255,0.9)"
    arrow_color = "rgba(0,70,220,1)"

    lw = 1 + 0.55 * current_mag
    aw = max(4, int(1 + current_mag))

    sgn = current_sign(current_dir)

    # Draw field circles at several heights along the wire.
    zvals = np.linspace(-0.4 * length, 0.4 * length, 6)

    # Draw circles with different radii.
    radii = [0.45, 0.9, 1.35]

    for z0 in zvals:
        for r in radii:

            t = np.linspace(0, 2 * np.pi, 300)

            # Reverse the circle direction for negative current.
            if sgn < 0:
                t = t[::-1]

            x = r * np.cos(t)
            y = r * np.sin(t)
            z = np.full_like(x, z0)

            pts = np.column_stack([x, y, z])

            fig.add_trace(go.Scatter3d(
                x=x,
                y=y,
                z=z,
                mode="lines",
                line=dict(color=line_color, width=lw),
                showlegend=False,
                hoverinfo="skip"
            ))

            add_arrow_on_curve(fig, pts, color=arrow_color, size=0.16, width=aw)


# ============================================================
# Magnetic field for a circular loop
# ============================================================

def add_circle_field(fig, current_mag, current_dir, radius, box_size):
    """
    Draw magnetic field lines for a circular current loop.

    We compute a field line in one plane and rotate it around the z-axis.
    This works because the circular loop is rotationally symmetric.
    """

    current = current_mag * current_dir
    wire_x, wire_y, wire_z = make_wire("circle", radius=radius, n=400)

    line_color = "rgba(0,140,255,0.9)"
    arrow_color = "rgba(0,70,220,1)"

    lw = 1 + 0.45 * current_mag
    aw = max(4, int(1 + current_mag))

    seed_pairs = []

    # Starting positions for field lines in the xz-plane.
    r_values = [0.28 * radius, 0.55 * radius, 0.78 * radius]
    z_values = [0.45 * radius, 0.85 * radius, -0.45 * radius, -0.85 * radius]

    for r in r_values:
        for z in z_values:
            seed_pairs.append((r, z))

    # Rotate each field line around the z-axis.
    phis = np.linspace(0, 2 * np.pi, 8, endpoint=False)

    for sx, sz in seed_pairs:

        # Trace forward along the magnetic field.
        pts_f = trace_meridian_line(
            sx, sz,
            wire_x, wire_y, wire_z,
            current=current,
            step=0.038,
            nsteps=300,
            box=box_size
        )

        # Trace backward along the same field line.
        pts_b = trace_meridian_line(
            sx, sz,
            wire_x, wire_y, wire_z,
            current=-current,
            step=0.038,
            nsteps=300,
            box=box_size
        )

        if len(pts_f) < 20 or len(pts_b) < 20:
            continue

        # Combine backward and forward pieces into one long curve.
        curve = np.vstack([pts_b[::-1], pts_f[1:]])

        for phi in phis:
            pts = rotate_xz_curve(curve, phi)

            fig.add_trace(go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode="lines",
                line=dict(color=line_color, width=lw),
                showlegend=False,
                hoverinfo="skip"
            ))

            add_arrow_on_curve(fig, pts, color=arrow_color, size=0.15, width=aw)


# ============================================================
# Magnetic field for a square loop
# ============================================================

def add_square_field(fig, current_mag, current_dir, radius, box_size):
    """
    Draw approximate magnetic field lines for a square current loop.

    This is not as perfectly symmetric as a circular loop,
    but it is useful for showing that any current loop produces
    a loop-like magnetic field.
    """

    current = current_mag * current_dir
    wire_x, wire_y, wire_z = make_wire("square", radius=radius, n=400)

    line_color = "rgba(0,140,255,0.9)"
    arrow_color = "rgba(0,70,220,1)"

    lw = 1 + 0.45 * current_mag
    aw = max(4, int(1 + current_mag))

    a = radius

    # Starting points for field lines.
    seeds = [
        [0.65 * a, 0.0, 0.55 * a],
        [-0.65 * a, 0.0, 0.55 * a],
        [0.0, 0.65 * a, 0.55 * a],
        [0.0, -0.65 * a, 0.55 * a],
        [0.65 * a, 0.0, -0.55 * a],
        [-0.65 * a, 0.0, -0.55 * a],
        [0.0, 0.65 * a, -0.55 * a],
        [0.0, -0.65 * a, -0.55 * a],
    ]

    for s in seeds:

        pts_f = trace_generic_field_line(
            s, wire_x, wire_y, wire_z, current,
            step=0.035,
            nsteps=280,
            box=box_size
        )

        pts_b = trace_generic_field_line(
            s, wire_x, wire_y, wire_z, -current,
            step=0.035,
            nsteps=280,
            box=box_size
        )

        if len(pts_f) > 1 and len(pts_b) > 1:
            pts = np.vstack([pts_b[::-1], pts_f[1:]])
        elif len(pts_f) > 1:
            pts = pts_f
        elif len(pts_b) > 1:
            pts = pts_b
        else:
            continue

        fig.add_trace(go.Scatter3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            mode="lines",
            line=dict(color=line_color, width=lw),
            showlegend=False,
            hoverinfo="skip"
        ))

        if len(pts) > 20:
            add_arrow_on_curve(fig, pts, color=arrow_color, size=0.15, width=aw)


# ============================================================
# Magnetic field for an arc
# ============================================================

def add_arc_field(fig, current_mag, current_dir, radius, box_size):
    """
    Draw clean local magnetic field circles around an arc.

    A real arc has a complicated magnetic field.
    For a teaching visualization, we draw local circular loops
    around several small current elements along the arc.

    This shows the right-hand-rule idea clearly.
    """

    sgn = current_sign(current_dir)

    line_color = "rgba(0,140,255,0.9)"
    arrow_color = "rgba(0,70,220,1)"

    lw = 1 + 0.45 * current_mag
    aw = max(4, int(1 + current_mag))

    # Choose several points along the arc.
    arc_angles = np.linspace(-1.05, 1.05, 5)

    for th in arc_angles:

        # Center of a small local field loop.
        center = np.array([
            radius * np.cos(th),
            radius * np.sin(th),
            0.0
        ])

        # Radial direction from origin to the arc point.
        radial = np.array([
            np.cos(th),
            np.sin(th),
            0.0
        ])

        # z direction.
        zhat = np.array([0.0, 0.0, 1.0])

        # Draw two small loops around each current element.
        for loop_r in [0.18 * radius, 0.30 * radius]:

            t = np.linspace(0, 2 * np.pi, 160)

            pts = []

            for a in t:

                # Circle in a plane perpendicular to the local current direction.
                p = center + loop_r * (np.cos(a) * radial + np.sin(a) * zhat)
                pts.append(p)

            pts = np.array(pts)

            # Reverse direction depending on the current direction.
            if sgn > 0:
                pts = pts[::-1]

            fig.add_trace(go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode="lines",
                line=dict(color=line_color, width=lw),
                showlegend=False,
                hoverinfo="skip"
            ))

            add_arrow_on_curve(fig, pts, color=arrow_color, size=0.10, width=aw)


# ============================================================
# Magnetic field for a solenoid
# ============================================================

def add_solenoid_field(fig, current_mag, current_dir, radius, length, turns, box_size):
    """
    Draw magnetic field lines for a solenoid.

    A solenoid is a coil of wire.
    The field is strong and mostly straight inside the coil.
    Outside the coil, the field loops back around.
    """

    current = current_mag * current_dir

    # Build the actual solenoid wire as a helix along the x-axis.
    t = np.linspace(0, 2 * np.pi * turns, 500)
    wire_x = np.linspace(-length / 2, length / 2, 500)
    wire_y = radius * np.cos(t)
    wire_z = radius * np.sin(t)

    line_color = "rgba(0,140,255,0.9)"
    arrow_color = "rgba(0,70,220,1)"

    lw = 1 + 0.45 * current_mag
    aw = max(4, int(1 + current_mag))

    # Seeds outside and inside the solenoid.
    seed_pairs = [
        (-0.20 * length, 0.65 * radius),
        (0.20 * length, 0.65 * radius),
        (-0.20 * length, -0.65 * radius),
        (0.20 * length, -0.65 * radius),
    ]

    mid_x_values = np.linspace(-0.25 * length, 0.25 * length, 3)

    mid_z_values = [
        0.25 * radius,
        0.50 * radius,
        -0.25 * radius,
        -0.50 * radius,
    ]

    for sx in mid_x_values:
        for sz in mid_z_values:
            seed_pairs.append((sx, sz))

    # Rotate field lines around the x-axis.
    phis = np.linspace(0, 2 * np.pi, 4, endpoint=False)

    for sx, sz in seed_pairs:

        pts_f = trace_meridian_line(
            sx, sz,
            wire_x, wire_y, wire_z,
            current=current,
            step=0.040,
            nsteps=280,
            box=box_size
        )

        pts_b = trace_meridian_line(
            sx, sz,
            wire_x, wire_y, wire_z,
            current=-current,
            step=0.040,
            nsteps=280,
            box=box_size
        )

        if len(pts_f) < 20 or len(pts_b) < 20:
            continue

        curve = np.vstack([pts_b[::-1], pts_f[1:]])

        for phi in phis:

            pts = rotate_xz_curve_about_x(curve, phi)

            fig.add_trace(go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode="lines",
                line=dict(color=line_color, width=lw),
                showlegend=False,
                hoverinfo="skip"
            ))

            add_arrow_on_curve(fig, pts, color=arrow_color, size=0.10, width=aw)


# ============================================================
# App layout
# ============================================================

app.layout = html.Div(
    style={"fontFamily": "Arial", "maxWidth": "1450px", "margin": "0 auto"},
    children=[

        html.H2("Biot-Savart Law Explorer"),

        html.Div(
            style={"display": "flex", "gap": "20px"},
            children=[

                # Left control panel
                html.Div(
                    style={
                        "width": "360px",
                        "padding": "16px",
                        "border": "1px solid #ddd",
                        "borderRadius": "12px",
                        "backgroundColor": "#f8f8f8",
                    },
                    children=[

                        html.Label("Wire shape"),

                        dcc.Dropdown(
                            id="shape",
                            options=[
                                {"label": "Straight wire", "value": "straight"},
                                {"label": "Circular loop", "value": "circle"},
                                {"label": "Arc", "value": "arc"},
                                {"label": "Square loop", "value": "square"},
                                {"label": "Solenoid", "value": "solenoid"},
                            ],
                            value="straight",
                            clearable=False,
                        ),

                        html.Br(),

                        html.Label("Current magnitude"),

                        dcc.Slider(
                            id="current_mag",
                            min=0.5,
                            max=10,
                            step=0.5,
                            value=3.5,
                            marks={i: str(i) for i in range(1, 11)},
                        ),

                        html.Br(),

                        html.Label("Current direction"),

                        dcc.Dropdown(
                            id="current_dir",
                            options=[
                                {"label": "Positive", "value": 1},
                                {"label": "Negative", "value": -1},
                            ],
                            value=-1,
                            clearable=False,
                        ),

                        html.Br(),

                        html.Div(
                            id="radius-box",
                            children=[
                                html.Label("Radius / half-side"),
                                dcc.Slider(
                                    id="radius",
                                    min=0.5,
                                    max=2.0,
                                    step=0.1,
                                    value=1.0,
                                    marks={0.5: "0.5", 1.0: "1", 2.0: "2"},
                                ),
                            ],
                        ),

                        html.Br(),

                        html.Div(
                            id="length-box",
                            children=[
                                html.Label("Height / wire length"),
                                dcc.Slider(
                                    id="length",
                                    min=1.0,
                                    max=6.0,
                                    step=0.5,
                                    value=2.5,
                                    marks={1: "1", 3: "3", 6: "6"},
                                ),
                            ],
                        ),

                        html.Br(),

                        html.Div(
                            id="turns-box",
                            children=[
                                html.Label("Solenoid turns"),
                                dcc.Slider(
                                    id="turns",
                                    min=3,
                                    max=12,
                                    step=1,
                                    value=8,
                                    marks={3: "3", 6: "6", 9: "9", 12: "12"},
                                ),
                            ],
                        ),

                        html.Br(),

                        html.Label("View size"),

                        dcc.Slider(
                            id="box_size",
                            min=1.5,
                            max=8.0,
                            step=0.5,
                            value=4,
                            marks={2: "2", 4: "4", 6: "6", 8: "8"},
                        ),
                    ],
                ),

                # Right plot area
                html.Div(
                    style={"flex": "1"},
                    children=[
                        dcc.Graph(id="field-plot", style={"height": "85vh"})
                    ],
                ),
            ],
        ),
    ],
)


# ============================================================
# Show or hide controls
# ============================================================

@app.callback(
    Output("radius-box", "style"),
    Output("length-box", "style"),
    Output("turns-box", "style"),
    Input("shape", "value"),
)
def show_hide_controls(shape):
    """
    This callback decides which sliders should be visible.

    Example:
        Straight wire needs length, but not radius.
        Circle needs radius, but not length.
        Solenoid needs radius, length, and turns.
    """

    show = {"display": "block"}
    hide = {"display": "none"}

    if shape == "straight":
        return hide, show, hide

    if shape == "circle":
        return show, hide, hide

    if shape == "arc":
        return show, hide, hide

    if shape == "square":
        return show, hide, hide

    if shape == "solenoid":
        return show, show, show

    return show, show, hide


# ============================================================
# Main plotting callback
# ============================================================

@app.callback(
    Output("field-plot", "figure"),
    Input("shape", "value"),
    Input("current_mag", "value"),
    Input("current_dir", "value"),
    Input("box_size", "value"),
    Input("radius", "value"),
    Input("length", "value"),
    Input("turns", "value"),
)
def update_plot(shape, current_mag, current_dir, box_size, radius, length, turns):
    """
    This is the main function that updates the plot.

    Every time the user changes a dropdown or slider,
    Dash automatically calls this function again.
    """

    fig = go.Figure()

    # Build the red wire.
    if shape == "solenoid":

        # For solenoid, we use a helix along the x-axis.
        t = np.linspace(0, 2 * np.pi * turns, 500)
        wire_x = np.linspace(-length / 2, length / 2, 500)
        wire_y = radius * np.cos(t)
        wire_z = radius * np.sin(t)

    else:

        # For all other shapes, use make_wire.
        wire_x, wire_y, wire_z = make_wire(shape, radius, length, n=400)

    # Draw magnetic field lines depending on the selected shape.
    if shape == "straight":
        add_straight_wire_field(fig, current_mag, current_dir, length)

    elif shape == "circle":
        add_circle_field(fig, current_mag, current_dir, radius, box_size)

    elif shape == "arc":
        add_arc_field(fig, current_mag, current_dir, radius, box_size)

    elif shape == "square":
        add_square_field(fig, current_mag, current_dir, radius, box_size)

    elif shape == "solenoid":
        add_solenoid_field(fig, current_mag, current_dir, radius, length, turns, box_size)

    # Draw the current-carrying wire in red.
    fig.add_trace(go.Scatter3d(
        x=wire_x,
        y=wire_y,
        z=wire_z,
        mode="lines",
        line=dict(width=8, color="red"),
        name="Wire"
    ))

    # Draw the black arrow showing current direction.
    add_wire_current_arrow(fig, shape, wire_x, wire_y, wire_z, current_dir)

    # Text for plot title.
    direction_text = "Positive" if current_sign(current_dir) > 0 else "Negative"

    # Default camera position.
    camera_eye = dict(x=1.55, y=1.55, z=1.2)

    # Solenoid looks better from a different angle.
    if shape == "solenoid":
        camera_eye = dict(x=0.18, y=-3.2, z=0.25)

    # Final plot formatting.
    fig.update_layout(
        margin=dict(l=0, r=0, t=30, b=0),
        title=f"{shape.replace('_', ' ').title()} | Current = {current_mag} | Direction = {direction_text}",
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode="cube",
            xaxis=dict(range=[-box_size, box_size]),
            yaxis=dict(range=[-box_size, box_size]),
            zaxis=dict(range=[-box_size, box_size]),
            camera=dict(eye=camera_eye),
        ),
    )

    return fig


# ============================================================
# Run the app
# ============================================================

if __name__ == "__main__":
    app.run(debug=True)